# Lesson 6: Satellite Imagery Basics

**Planning question: What does the city look like from space?**

Satellites photograph the whole Earth every few days. For planners, this means an up-to-date view of any city,
including areas with poor maps. In this notebook you will:

1. Search a catalogue of free **Sentinel-2** images
2. Pick a clear, cloud-free image of Bengaluru
3. Make true colour and false colour pictures
4. See how water, trees and buildings reflect light differently

Part of [RS & GIS for Spatial Planners](https://soniadas123.github.io/rs-gis-spatial-planner/).

In [ ]:
!pip install -q pystac-client planetary-computer odc-stac rioxarray

## Step 1: Load the packages

- **pystac_client** searches satellite image catalogues (STAC)
- **planetary_computer** gives free access to Microsoft's Planetary Computer data
- **odc.stac** loads the images we pick as an **xarray** dataset, which is a stack of grids
- **rioxarray** adds map tools such as saving to GeoTIFF

In [ ]:
import pystac_client
import planetary_computer
import odc.stac
import rioxarray
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer

## Settings

`BBOX` is the box we want, written as `[west, south, east, north]` in degrees. This one covers central Bengaluru.
Images from January to March are used because that is the dry season, with the fewest clouds.

In [ ]:
BBOX = [77.50, 12.90, 77.68, 13.05]
DATES = "2024-01-01/2024-03-31"
UTM_CRS = "EPSG:32643"

## Step 2: Search the catalogue

**Sentinel-2** is a pair of European satellites. They pass over every place about every 5 days and take images at 10 m resolution.
We ask the catalogue for images that cover our box, fall within our dates and have less than 10 percent cloud.

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime=DATES,
    query={"eo:cloud_cover": {"lt": 10}},
)
items = search.item_collection()
print("Images found:", len(items))

Each result is one **tile**, a 100 km square of one image. Let us list them.

In [ ]:
table = pd.DataFrame([{
    "date": item.datetime.date(),
    "tile": item.properties["s2:mgrs_tile"],
    "cloud_%": round(item.properties["eo:cloud_cover"], 1),
} for item in items])
table.sort_values("date")

## Step 3: Pick the clearest date

Bengaluru sits where two tiles meet, so we need a date where the tiles together cover our whole box.
This function checks the coverage of each date and returns the tiles for the clearest one.

In [ ]:
from shapely.geometry import shape, box
from shapely import union_all


def clearest_date_items(items, bbox):
    """Return the items from the least cloudy date that fully covers bbox."""
    by_date = {}
    for item in items:
        by_date.setdefault(item.datetime.date(), []).append(item)

    area = box(*bbox)
    options = []
    for date, date_items in by_date.items():
        footprint = union_all([shape(i.geometry) for i in date_items])
        if footprint.contains(area):
            cloud = np.mean([i.properties["eo:cloud_cover"] for i in date_items])
            options.append((cloud, date, date_items))

    if not options:
        raise ValueError("No date covers the whole box. Try a longer date range or a higher cloud limit.")
    cloud, date, date_items = min(options, key=lambda option: option[0])
    print(f"Using {date} ({len(date_items)} tiles, {cloud:.1f}% cloud)")
    return date_items


best_items = clearest_date_items(items, BBOX)

## Step 4: Load the image

A satellite image is made of **bands**. Each band records one colour of light. We load five of them:

| Band | Light | Wavelength (nm) |
|---|---|---|
| B02 | Blue | 490 |
| B03 | Green | 560 |
| B04 | Red | 665 |
| B08 | Near-infrared (NIR) | 842 |
| B11 | Short-wave infrared (SWIR) | 1610 |

We load the bands at 20 m pixels, in UTM, clipped to our box.

In [ ]:
ds = odc.stac.load(
    best_items,
    bands=["B02", "B03", "B04", "B08", "B11"],
    bbox=BBOX,
    crs=UTM_CRS,
    resolution=20,
    groupby="solar_day",
).isel(time=0)
ds

## Step 5: Convert to reflectance

The pixel values are stored as whole numbers. To get **reflectance**, the share of sunlight bounced back from 0 to 1,
we subtract an offset of 1000 (for images processed since 2022) and divide by 10,000. Zero means "no data".

In [ ]:
def to_reflectance(ds, item):
    offset = 1000 if float(item.properties["s2:processing_baseline"]) >= 4 else 0
    ds = ds.where(ds > 0)
    return ((ds - offset) / 10000).clip(0, 1)


refl = to_reflectance(ds, best_items[0])
refl["B04"].plot(cmap="Reds", size=6, robust=True)
plt.title("Red band (B04) reflectance")
plt.show()

## Step 6: True colour image

Combining the red, green and blue bands gives the picture our eyes would see.
`robust=True` stretches the colours so the image is not too dark.

In [ ]:
true_colour = refl[["B04", "B03", "B02"]].to_array("band")
true_colour.plot.imshow(rgb="band", robust=True, size=9)
plt.title("Sentinel-2 true colour: central Bengaluru")
plt.show()

## Step 7: False colour image

Plants reflect a lot of **near-infrared** light, which we cannot see.
If we show NIR as red, healthy vegetation lights up bright red. Look for Cubbon Park, Lalbagh and the tree-lined streets.

In [ ]:
false_colour = refl[["B08", "B04", "B03"]].to_array("band")
false_colour.plot.imshow(rgb="band", robust=True, size=9)
plt.title("False colour (NIR-Red-Green): vegetation is red")
plt.show()

A third combination, SWIR-NIR-Red, makes built-up areas purple and bare land brown.

In [ ]:
urban_colour = refl[["B11", "B08", "B04"]].to_array("band")
urban_colour.plot.imshow(rgb="band", robust=True, size=9)
plt.title("Urban false colour (SWIR-NIR-Red)")
plt.show()

## Step 8: Spectral signatures

Every surface reflects light in its own pattern, called its **spectral signature**. This is the basis of all remote sensing.
We read the pixel values at three known places and compare them.

In [ ]:
places = {
    "Water (Ulsoor Lake)": (12.9830, 77.6200),
    "Trees (Cubbon Park)": (12.9763, 77.5929),
    "Buildings (Majestic)": (12.9767, 77.5713),
}
wavelengths = [490, 560, 665, 842, 1610]
to_utm = Transformer.from_crs("EPSG:4326", UTM_CRS, always_xy=True)

fig, ax = plt.subplots(figsize=(8, 5))
for name, (lat, lon) in places.items():
    x, y = to_utm.transform(lon, lat)
    pixel = refl.sel(x=x, y=y, method="nearest")
    values = [float(pixel[b]) for b in ["B02", "B03", "B04", "B08", "B11"]]
    ax.plot(wavelengths, values, marker="o", label=name)

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance")
ax.legend()
ax.set_title("Spectral signatures")
plt.show()

Notice that:

- **Water** absorbs almost all near-infrared light, so its values are low.
- **Trees** jump up sharply in the near-infrared.
- **Buildings** are fairly flat across all bands and higher in the short-wave infrared.

Lesson 7 uses these differences to map green cover and built-up areas.

## Step 9: Save the image as a GeoTIFF

A GeoTIFF is an image file that knows its location, so it lines up correctly when you open it in QGIS.

In [ ]:
true_colour.rio.to_raster("bengaluru_true_colour.tif")
print("Saved bengaluru_true_colour.tif")

## What this means for planning

- Free satellite images give an up-to-date base layer for any site, even where maps are poor.
- False colour images make green cover, water bodies and construction easy to spot.
- At 10 to 20 m, Sentinel-2 shows neighbourhoods and large buildings but not individual houses.
  Detailed site work needs higher resolution imagery or drone surveys.

## Try it yourself

1. Change `DATES` to `"2019-01-01/2019-03-31"` and compare the images. Can you spot new construction?
2. Move `BBOX` to the airport area: `[77.65, 13.15, 77.76, 13.25]`.
3. Add a point for Bellandur Lake `(12.9350, 77.6680)` to `places`. Note that it lies just outside the current box, so make `BBOX` bigger first.